# PhysioGraph: biological follow-up of SpO2 instability

**Active scope: MIMIC and eICU only.** The active experiment is the final
section, "Circulatory context of SpO2 instability", which develops the
existing SpO2-myocardial injury/decompensation findings. Earlier phosphate
cells retain the completed, inconclusive experiment for auditability.

Run all cells in order. Uses the original credentialed MIMIC/eICU files in
**My Drive/Data**, never toy or simulated patients. In Colab, mount your own
Drive when prompted. No patient data are transmitted to third-party services.
The delivered execution, when present, is **local**, not Colab.

The first candidate tests phosphate rise during lactate normalization against
subsequent hospital mortality. A significant association is insufficient to
establish a new mechanism, treatment benefit, or a paradigm-shifting result.
This notebook deliberately reports inadequate support and rejected hypotheses.

# Biological discovery protocol: residual danger after lactate improvement

User-directed scope amendment, 2026-09-05 UTC. The objective is a substantial
biological finding relevant to heart-failure mortality, comparable in clinical
importance to lactate-informed shock staging. Measurement-method results do
not satisfy this objective. No result is currently claimed.

## First falsifiable candidate

Test whether an early rise in circulating phosphate identifies ongoing
physiological deterioration in HF patients whose lactate normalizes, beyond
renal dysfunction, initial phosphate, and the lactate trajectory. The
biological possibilities include impaired renal elimination, cellular injury,
acid-base shifts and administered phosphate; serum phosphate does not directly
measure ATP depletion. Distinguishing those explanations is part of the
research, not an inference to make from a regression coefficient.

Phosphate alone is NOT novel: a 2025 MIMIC-IV cardiogenic-shock study already
examined its nonlinear mortality association
(https://www.medrxiv.org/content/10.1101/2025.05.20.25327987v1.full), and a 2023
cardiac-arrest study compared phosphate with lactate
(https://pmc.ncbi.nlm.nih.gov/articles/PMC10177342/). Normal lactate despite low
cardiac output in advanced HF is also established
(https://pubmed.ncbi.nlm.nih.gov/33734459/). This candidate concerns the joint
early trajectory, conditional on renal trajectory, with cross-database
replication and a decision-relevant residual mortality risk. That specificity
does not by itself establish novelty or clinical importance.

## Frozen first experiment

- Population: adults with explicitly documented HF, first ICU unit/stay per
  hospital encounter, admitted to ICU within 24 hours of hospital admission.
  Alive and still in the ICU at 12 hours. This deliberately excludes late ICU
  shock cases rather than using a discharge shock diagnosis as an early
  predictor. Retrospective discharge-code HF ascertainment in MIMIC is a
  cohort limitation, not evidence of real-time HF recognition.
- Baseline: first valid lab in ICU hours [0,4). Follow-up: last valid lab in
  [8,12). No value at or after 12 hours enters predictors. Actual collection
  times and reporting/revision times are retained; delayed availability
  requires a sensitivity analysis before any bedside prediction claim.
- Primary risk set: observed baseline lactate >=2 mmol/L and follow-up
  lactate <2 mmol/L, with phosphate and creatinine measured in both windows.
  Exposure: phosphate increase >=0.5 mg/dL. Reference: increase <0.5 mg/dL.
  Thresholds are investigator-specified before reading these associations.
- Primary outcome: death before hospital discharge, after the 12-hour
  landmark. Unknown discharge status is unavailable, never a survivor.
  Hospital mortality is used because it is shared across both databases.
- Primary contrast: adjusted mortality association of phosphate rise within
  lactate normalizers. Fit modified Poisson with patient-clustered robust
  standard errors, adjusting for age, sex, log1p baseline lactate, log1p
  follow-up lactate, log1p baseline phosphate, log1p baseline creatinine and
  creatinine change. Explicit complete-case denominators; >=100 deaths and
  >=20 deaths in each exposure arm are required to grade a fitted result.
  No coefficient selection, threshold optimization or outcome-driven covariates.
- eICU is discovery; transfer identical definitions and model specification to
  MIMIC. Both databases have been explored in earlier project work; this is
  a newly specified analysis, not an untouched prospectively held-out cohort.
- Secondary falsification: repeat among those with baseline creatinine <2
  mg/dL and creatinine rise <0.3 mg/dL; analyze continuous phosphate change;
  report four joint lactate/phosphate trajectory groups in the baseline
  hyperlactatemia population. Secondary p-values do not rescue a failed primary.
- RRT, phosphate supplementation, cardiac arrest, infection, respiratory
  interventions, pH and baseline organ injury require source-grounded
  sensitivity controls before a biological or actionable claim. Missing
  treatment information is not absence of treatment.

## Sequential decision gates

1. Audit assay identity/units, patient grouping, chronology, denominators and
   missingness from the ORIGINAL databases. No ignored repository caches.
2. Execute the locked primary contrast and publish every result, including
   sparse/null/reversed estimates. Stop promoting this candidate if it fails
   replication or renal/treatment sensitivity. Do not select the best result
   from additional time windows.
3. Only if it survives, compare patient/site-held-out mortality prediction
   with lactate and a faithfully reconstructable SCAI benchmark on the same
   patients. A partial lab-only adjustment is not SCAI adjustment. Require
   calibration and decision-curve utility, not AUROC alone.
4. For prioritization, seek >=5 percentage points adjusted absolute residual
   risk and a replicated risk ratio whose lower 95% limit exceeds 1.25.
   These are research prioritization thresholds, not validated clinical
   decision cutoffs or proof of a practice-changing discovery.
5. An observational association cannot prove phosphate manipulation or any
   other response reduces mortality. Mechanistic discrimination, genuinely
   independent clinical replication and prospective intervention evidence
   remain necessary for the user's ultimate target. Never call that target
   achieved because one retrospective p-value is small.

## Reproducibility

One executable `PhysioGraph_Biological_Discovery.ipynb` will contain the raw
extraction, checks, analyses and outputs. Local execution is labeled local.
Patient-level intermediate files belong under the original Drive Data area,
outside the repository; repository deliverables contain aggregate results.
Preserve original study artifacts and pre-existing edits. Log failed runs.
Keep this protocol and its hash with each output and record further
amendments before their analyses. No fake, simulated or precomputed data may
stand in for these clinical experiments.

In [1]:
import importlib.util, subprocess, sys
requirements={'duckdb':'duckdb==1.4.3','numpy':'numpy==2.2.6',
    'pandas':'pandas==2.2.3','scipy':'scipy==1.15.3','statsmodels':'statsmodels==0.14.4'}
missing=[p for m,p in requirements.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import os, json, hashlib, time, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE=Path('/content/drive/MyDrive')
else:
    DRIVE=Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
ENVIRONMENT='google_colab' if IN_COLAB else 'local'
RAW={d:DRIVE/'Data'/d/'Full' for d in ['eICU','MIMIC']}
assert all(p.is_dir() for p in RAW.values()), 'Expected original databases under My Drive/Data'
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
PRIVATE.mkdir(exist_ok=True)
OUT=Path.cwd()/'research'/'biological_discovery'
OUT.mkdir(parents=True,exist_ok=True)
START=datetime.now(timezone.utc).isoformat()
SEED=20260905
def write_json(name,value):
    p=OUT/name
    p.write_text(json.dumps(value,indent=2,allow_nan=False,default=str)+'\n')
def status(stage,state='running',**kwargs):
    write_json('run_status.json',dict(status=state,stage=stage,environment=ENVIRONMENT,
        started_utc=START,updated_utc=datetime.now(timezone.utc).isoformat(),**kwargs))
    print(stage,state,flush=True)
status('setup')
print('Data source:',DRIVE/'Data')
print('Patient-level intermediates:',PRIVATE)
con=duckdb.connect(str(PRIVATE/'clinical.duckdb'))
con.execute("SET threads=2")
con.execute("SET memory_limit='2GB'")
source_manifest={}
def sqlstr(x): return "'"+str(x).replace("'","''")+"'"
def rawview(dataset,name):
    p=RAW[dataset]/(name+'.csv')
    assert p.is_file(), str(p)
    s=p.stat()
    source_manifest[dataset+'/'+name]=dict(path=str(p),bytes=s.st_size,mtime_ns=s.st_mtime_ns)
    return f"read_csv({sqlstr(p)},header=true,all_varchar=true,sample_size=10000,strict_mode=true)"

setup running
Data source: /Users/admin/Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive/Data
Patient-level intermediates: /Users/admin/Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive/Data/PhysioGraph_Biological_Discovery_20260905


In [2]:
PROTOCOL_SHA256='00e0b16738a169f146af74dca30caf72e905c675f153b3fbad98d1a404f7d27b'
write_json('protocol.json',dict(sha256=PROTOCOL_SHA256,frozen_before_analysis=True,seed=SEED))

## Cohorts and chronology

One ICU stay per hospital encounter. HF is explicitly coded; cardiomyopathy or
shock alone do not qualify. Cohort entry requires adult age, ICU admission
within 24 hours of hospital admission, and survival/in-ICU presence through
12 hours. MIMIC HF codes are retrospective discharge diagnoses. Repeat
hospitalizations retain patient identifiers for clustered uncertainty.

In [3]:
status('eICU cohort')
p=rawview('eICU','patient'); dx=rawview('eICU','diagnosis'); adx=rawview('eICU','admissionDx')
hf_regex=r'(^|[|/ ,])(congestive heart failure|heart failure|chf)([|/ ,]|$)'
con.execute(f"""
CREATE OR REPLACE TABLE eicu_cohort AS
WITH hf AS (
 SELECT DISTINCT patientunitstayid FROM {dx}
 WHERE regexp_matches(lower(diagnosisstring),{sqlstr(hf_regex)})
 UNION SELECT DISTINCT patientunitstayid FROM {adx}
 WHERE regexp_matches(lower(coalesce(admitdxname,'')||' '||coalesce(admitdxtext,'')),{sqlstr(hf_regex)})
), ranked AS (
 SELECT *,row_number() OVER(PARTITION BY patienthealthsystemstayid
 ORDER BY try_cast(hospitaladmitoffset AS DOUBLE) DESC,try_cast(patientunitstayid AS BIGINT)) rn FROM {p}
)
SELECT try_cast(patientunitstayid AS BIGINT) stay_id,uniquepid patient_id,
 patienthealthsystemstayid encounter_id,hospitalid site,
 try_cast(replace(age,'>','') AS DOUBLE) age,
 CASE lower(gender) WHEN 'male' THEN 1.0 WHEN 'female' THEN 0.0 END male,
 CASE lower(hospitaldischargestatus) WHEN 'expired' THEN 1 WHEN 'alive' THEN 0 END mortality,
 try_cast(unitdischargeoffset AS DOUBLE) unit_end,
 try_cast(hospitaldischargeoffset AS DOUBLE) hospital_end
FROM ranked WHERE rn=1 AND patientunitstayid IN(SELECT patientunitstayid FROM hf)
 AND try_cast(replace(age,'>','') AS DOUBLE)>=18
 AND try_cast(hospitaladmitoffset AS DOUBLE) BETWEEN -1440 AND 0
 AND try_cast(unitdischargeoffset AS DOUBLE)>720
 AND try_cast(hospitaldischargeoffset AS DOUBLE)>720
 AND uniquepid IS NOT NULL
""")
status('MIMIC cohort')
dx=rawview('MIMIC','diagnoses_icd'); p=rawview('MIMIC','patients')
a=rawview('MIMIC','admissions'); i=rawview('MIMIC','icustays')
con.execute(f"""
CREATE OR REPLACE TABLE mimic_cohort AS
WITH hf AS (
 SELECT DISTINCT hadm_id FROM {dx} WHERE
 (icd_version='9' AND starts_with(icd_code,'428')) OR
 (icd_version='10' AND starts_with(upper(icd_code),'I50'))
), ranked AS (
 SELECT *,row_number() OVER(PARTITION BY hadm_id ORDER BY intime,stay_id) rn FROM {i}
)
SELECT try_cast(i.stay_id AS BIGINT) stay_id,i.subject_id patient_id,i.hadm_id encounter_id,
 'MIMIC' site,try_cast(p.anchor_age AS DOUBLE)+year(try_cast(a.admittime AS TIMESTAMP))-
 try_cast(p.anchor_year AS DOUBLE) age,
 CASE p.gender WHEN 'M' THEN 1.0 WHEN 'F' THEN 0.0 END male,
 try_cast(a.hospital_expire_flag AS INTEGER) mortality,try_cast(i.intime AS TIMESTAMP) intime,
 epoch(try_cast(i.outtime AS TIMESTAMP)-try_cast(i.intime AS TIMESTAMP))/60 unit_end,
 epoch(try_cast(a.dischtime AS TIMESTAMP)-try_cast(i.intime AS TIMESTAMP))/60 hospital_end
FROM ranked i JOIN {a} a ON i.hadm_id=a.hadm_id AND i.subject_id=a.subject_id
 JOIN {p} p ON i.subject_id=p.subject_id
WHERE i.rn=1 AND i.hadm_id IN(SELECT hadm_id FROM hf)
 AND epoch(try_cast(i.intime AS TIMESTAMP)-try_cast(a.admittime AS TIMESTAMP))/60 BETWEEN 0 AND 1440
 AND age>=18 AND unit_end>720 AND hospital_end>720
 AND (a.deathtime IS NULL OR try_cast(a.deathtime AS TIMESTAMP)>try_cast(i.intime AS TIMESTAMP)+INTERVAL 12 HOUR)
""")
cohort_counts=[]
for d in ['eicu','mimic']:
    counts=con.execute(f'SELECT count(*) n,count(DISTINCT patient_id) patients,count(DISTINCT encounter_id) encounters FROM {d}_cohort').fetchone()
    assert counts[0]==counts[2] and counts[0]>100
    cohort_counts.append(dict(dataset=d,n=counts[0],patients=counts[1],encounters=counts[2]))
write_json('cohort_counts.json',cohort_counts)
print(pd.DataFrame(cohort_counts).to_string(index=False))

eICU cohort running
MIMIC cohort running
dataset     n  patients  encounters
   eicu 11596     10281       11596
  mimic 16659     12772       16659


## Extract actual lab observations

Stream original CSV tables; retain only eligible encounters and measurements
before 12 hours. Dictionary-derived MIMIC assays must be blood assays.
Unknown units are excluded and counted. Phosphate and creatinine use mg/dL;
lactate uses mmol/L. No missing measurement becomes a normal value.

In [4]:
status('eICU raw lab scan')
l=rawview('eICU','lab')
con.execute(f"""
CREATE OR REPLACE TABLE eicu_labs AS
SELECT try_cast(l.patientunitstayid AS BIGINT) stay_id,
 CASE lower(l.labname) WHEN 'lactate' THEN 'lactate' WHEN 'phosphate' THEN 'phosphate'
 WHEN 'phosphorus' THEN 'phosphate' WHEN 'creatinine' THEN 'creatinine' END marker,
 try_cast(l.labresultoffset AS DOUBLE) AS event_minute,try_cast(l.labresult AS DOUBLE) AS value,
 l.labname assay,l.labmeasurenamesystem unit,
 try_cast(l.labresultrevisedoffset AS DOUBLE) available_minute
FROM {l} l JOIN eicu_cohort c ON try_cast(l.patientunitstayid AS BIGINT)=c.stay_id
WHERE lower(l.labname) IN ('lactate','phosphate','phosphorus','creatinine')
 AND try_cast(l.labresultoffset AS DOUBLE)>=0 AND try_cast(l.labresultoffset AS DOUBLE)<720
""")
status('MIMIC raw lab scan')
d=rawview('MIMIC','d_labitems')
mapping=con.execute(f"""SELECT itemid,label,CASE lower(label)
 WHEN 'lactate' THEN 'lactate' WHEN 'phosphate' THEN 'phosphate'
 WHEN 'creatinine' THEN 'creatinine' WHEN 'creatinine, whole blood' THEN 'creatinine' END marker
 FROM {d} WHERE lower(fluid)='blood' AND lower(label) IN
 ('lactate','phosphate','creatinine','creatinine, whole blood')""").df()
assert set(mapping.marker)=={'lactate','phosphate','creatinine'}
con.register('assay_mapping',mapping)
write_json('mimic_assay_mapping.json',mapping.to_dict(orient='records'))
l=rawview('MIMIC','labevents')
con.execute(f"""
CREATE OR REPLACE TABLE mimic_labs AS
SELECT c.stay_id,m.marker,epoch(try_cast(l.charttime AS TIMESTAMP)-c.intime)/60 AS event_minute,
 try_cast(l.valuenum AS DOUBLE) AS value,l.itemid assay,l.valueuom unit,
 epoch(try_cast(l.storetime AS TIMESTAMP)-c.intime)/60 available_minute
FROM {l} l JOIN assay_mapping m ON l.itemid=m.itemid
 JOIN mimic_cohort c ON l.hadm_id=c.encounter_id AND l.subject_id=c.patient_id
WHERE try_cast(l.charttime AS TIMESTAMP)>=c.intime
 AND try_cast(l.charttime AS TIMESTAMP)<c.intime+INTERVAL 12 HOUR
""")
unit_audit=[]
for d in ['eicu','mimic']:
    tab=con.execute(f'SELECT marker,unit,count(*) n FROM {d}_labs GROUP BY marker,unit ORDER BY marker,unit').df()
    tab['dataset']=d
    unit_audit.extend(tab.where(pd.notna(tab),None).to_dict(orient='records'))
write_json('unit_audit.json',unit_audit)
print(pd.DataFrame(unit_audit).to_string(index=False))

eICU raw lab scan running
MIMIC raw lab scan running
    marker   unit     n dataset
creatinine  mg/dL 10092    eicu
   lactate mmol/L  4019    eicu
 phosphate  mg/dL  3527    eicu
creatinine  mg/dL 23424   mimic
   lactate mmol/L 19348   mimic
 phosphate  mg/dL 21150   mimic


In [5]:
status('feature construction and support audit')
frames={}; support=[]
for d in ['eicu','mimic']:
    con.execute(f"""
    CREATE OR REPLACE TABLE {d}_clean_labs AS
    SELECT stay_id,marker,event_minute,median(value) AS value,max(available_minute) AS available_minute
    FROM {d}_labs WHERE
    ((marker='lactate' AND lower(replace(unit,' ',''))='mmol/l' AND value BETWEEN 0.1 AND 30)
    OR (marker='phosphate' AND lower(replace(unit,' ',''))='mg/dl' AND value BETWEEN 0.2 AND 20)
    OR (marker='creatinine' AND lower(replace(unit,' ',''))='mg/dl' AND value BETWEEN 0.1 AND 20))
    GROUP BY stay_id,marker,event_minute
    """)
    c=con.execute(f'SELECT * FROM {d}_cohort').df().set_index('stay_id')
    for marker in ['lactate','phosphate','creatinine']:
        for label,lo,hi,ordering in [('b',0,240,'ASC'),('f',480,720,'DESC')]:
            t=con.execute(f"""SELECT stay_id,value,event_minute,available_minute FROM {d}_clean_labs
            WHERE marker={sqlstr(marker)} AND event_minute>={lo} AND event_minute<{hi}
            QUALIFY row_number() OVER(PARTITION BY stay_id ORDER BY event_minute {ordering})=1""").df().set_index('stay_id')
            c=c.join(t.rename(columns={k:f'{marker}_{label}_{k}' for k in t.columns}))
    for marker in ['lactate','phosphate','creatinine']:
        c[marker+'_delta']=c[marker+'_f_value']-c[marker+'_b_value']
    c['normalizer']=(c.lactate_b_value>=2)&(c.lactate_f_value<2)
    c['phosphate_rise']=(c.phosphate_delta>=0.5).astype(float).where(c.phosphate_delta.notna())
    required=[f'{m}_{w}_value' for m in ['lactate','phosphate','creatinine'] for w in ['b','f']]
    c['complete_labs']=c[required].notna().all(axis=1)
    c['available_by_landmark']=c[[f'{m}_{w}_available_minute' for m in ['lactate','phosphate','creatinine'] for w in ['b','f']]].lt(720).all(axis=1)
    assert c[[f'{m}_{w}_event_minute' for m in ['lactate','phosphate','creatinine'] for w in ['b','f']]].max().max()<720
    assert c.index.is_unique and c.encounter_id.is_unique
    for name,mask in [('cohort',pd.Series(True,index=c.index)),('paired_lactate',c.lactate_b_value.notna()&c.lactate_f_value.notna()),
                      ('normalizers',c.normalizer),('primary_complete_labs',c.normalizer&c.complete_labs)]:
        sub=c[mask]
        support.append(dict(dataset=d,stage=name,n=len(sub),known_outcomes=int(sub.mortality.notna().sum()),
            deaths=int(sub.mortality.sum()),sites=int(sub.site.nunique()),patients=int(sub.patient_id.nunique())))
    con.register('features_frame',c.reset_index())
    con.execute(f'CREATE OR REPLACE TABLE {d}_features AS SELECT * FROM features_frame')
    frames[d]=c
write_json('support_audit.json',support)
print(pd.DataFrame(support).to_string(index=False))

feature construction and support audit running
dataset                 stage     n  known_outcomes  deaths  sites  patients
   eicu                cohort 11596           11492    1228    200     10281
   eicu        paired_lactate   405             404     102     87       403
   eicu           normalizers    92              92      13     48        92
   eicu primary_complete_labs    18              18       2      9        18
  mimic                cohort 16659           16659    2209      1     12772
  mimic        paired_lactate  2540            2540     644      1      2432
  mimic           normalizers   506             506     102      1       497
  mimic primary_complete_labs   224             224      56      1       223


## Locked mortality comparison

Modified-Poisson regression with patient-clustered robust uncertainty. No
regression is graded when there are fewer than 100 deaths or fewer than 20
deaths in either phosphate arm. Descriptive risks remain visible. These
models do not yet adjust for SCAI, RRT or administered phosphate and cannot
support a new biological mechanism or a clinical action.

In [6]:
status('locked mortality models')
results=[]; risk_rows=[]
formula='mortality ~ phosphate_rise + age + male + np.log1p(lactate_b_value) + np.log1p(lactate_f_value) + np.log1p(phosphate_b_value) + np.log1p(creatinine_b_value) + creatinine_delta'
for d,c in frames.items():
    masks={'primary':c.normalizer&c.complete_labs,
           'renal_restriction':c.normalizer&c.complete_labs&(c.creatinine_b_value<2)&(c.creatinine_delta<.3),
           'available_by_landmark':c.normalizer&c.complete_labs&c.available_by_landmark}
    for label,mask in masks.items():
        s=c[mask].dropna(subset=['mortality','age','male','patient_id']).copy()
        arms=s.groupby('phosphate_rise').mortality.agg(['count','sum','mean'])
        for arm,row in arms.iterrows():
            risk_rows.append(dict(dataset=d,analysis=label,phosphate_rise=int(arm),n=int(row['count']),
                deaths=int(row['sum']),mortality_risk=float(row['mean'])))
        record=dict(dataset=d,analysis=label,n=len(s),patients=int(s.patient_id.nunique()),
            deaths=int(s.mortality.sum()),status='insufficient_support',biological_claim_ready=False)
        if len(arms)==2 and s.mortality.sum()>=100 and arms['sum'].min()>=20 and (arms['count']-arms['sum']).min()>=20:
            import statsmodels.api as sm
            import statsmodels.formula.api as smf
            model=smf.glm(formula,s,family=sm.families.Poisson()).fit(cov_type='cluster',cov_kwds={'groups':s.patient_id})
            b=float(model.params['phosphate_rise']); ci=model.conf_int().loc['phosphate_rise'].to_numpy()
            record.update(status='estimated',rr=float(np.exp(b)),ci95=np.exp(ci).tolist(),
                p=float(model.pvalues['phosphate_rise']),converged=bool(model.converged))
            record['support_note']='Requires treatment, acid-base, SCAI and multicenter controls before biological interpretation.'
        results.append(record)
write_json('locked_primary_results.json',results)
write_json('mortality_risks.json',risk_rows)
print(pd.DataFrame(risk_rows).to_string(index=False))
print(pd.DataFrame(results).to_string(index=False))
print('No significance or clinical claim is manufactured when data are sparse.')

locked mortality models running
dataset              analysis  phosphate_rise   n  deaths  mortality_risk
   eicu               primary               0  16       2        0.125000
   eicu               primary               1   2       0        0.000000
   eicu     renal_restriction               0  12       1        0.083333
   eicu     renal_restriction               1   2       0        0.000000
   eicu available_by_landmark               0  14       2        0.142857
   eicu available_by_landmark               1   2       0        0.000000
  mimic               primary               0 170      37        0.217647
  mimic               primary               1  54      19        0.351852
  mimic     renal_restriction               0  82      13        0.158537
  mimic     renal_restriction               1  26       6        0.230769
  mimic available_by_landmark               0 118      27        0.228814
  mimic available_by_landmark               1  43      14        0.325581
datase

In [7]:
status('provenance and output verification')
# Full source fingerprints are computed after extraction; source size/time
# stability is checked so changes during the scan fail the run.
for name,info in source_manifest.items():
    p=Path(info['path']); digest=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): digest.update(block)
    current=p.stat()
    assert current.st_size==info['bytes'] and current.st_mtime_ns==info['mtime_ns'], 'Source changed during run: '+name
    info['sha256']=digest.hexdigest()
    print('Fingerprinted',name,flush=True)
write_json('input_manifest.json',source_manifest)
outputs={p.name:dict(sha256=hashlib.sha256(p.read_bytes()).hexdigest(),bytes=p.stat().st_size)
    for p in sorted(OUT.iterdir()) if p.suffix=='.json' and p.name not in ['manifest.json','run_status.json']}
write_json('manifest.json',dict(environment=ENVIRONMENT,started_utc=START,
    finished_utc=datetime.now(timezone.utc).isoformat(),protocol_sha256=PROTOCOL_SHA256,
    python=sys.version,software={m:__import__(m).__version__ for m in ['duckdb','numpy','pandas','scipy','statsmodels']},
    outputs=outputs,synthetic_patients=0,source='original MIMIC/eICU under My Drive/Data',
    paradigm_shifting_result=False,clinical_benefit_established=False))
status('initial locked experiment','complete',goal_complete=False)
con.close()
print('The discovery goal remains open. Review the actual results before advancing or rejecting this candidate.')

provenance and output verification running
Fingerprinted eICU/patient
Fingerprinted eICU/diagnosis
Fingerprinted eICU/admissionDx
Fingerprinted MIMIC/diagnoses_icd
Fingerprinted MIMIC/patients
Fingerprinted MIMIC/admissions
Fingerprinted MIMIC/icustays
Fingerprinted eICU/lab
Fingerprinted MIMIC/d_labitems
Fingerprinted MIMIC/labevents
initial locked experiment complete
The discovery goal remains open. Review the actual results before advancing or rejecting this candidate.
